In [12]:
# ✅ Install dependencies
!pip install transformers sentencepiece gradio gTTS

import os
import tempfile
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
import gradio as gr
from gtts import gTTS

os.environ.pop("HUGGINGFACE_TOKEN", None)

# ✅ Load tokenizer and model
model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

print("✅ Model and tokenizer loaded!")

# 🌍 Translate function
def translate(text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )
    translated = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return translated[0]

# 🔊 Speak function
def speak(text, tgt_lang):
    if not text.strip():
        return None
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as fp:
        try:
            # Try TTS with target language
            tts = gTTS(text, lang=tgt_lang if tgt_lang in ["en", "hi", "fr", "de", "es"] else "en")
        except:
            # fallback to English if lang not supported
            tts = gTTS(text, lang="en")
        tts.save(fp.name)
        return fp.name

src_langs = ["en", "hi", "fr", "de", "es", "zh", "ja", "ko"]

# 🎛️ Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("# 🌍 Multi-Language Translator (M2M100) + 🔊 Speaker")

    with gr.Row():
        input_text = gr.Textbox(label="Enter text to translate")
    with gr.Row():
        src = gr.Dropdown(choices=src_langs, value="en", label="Source Language")
        tgt = gr.Dropdown(choices=src_langs, value="hi", label="Target Language")
    with gr.Row():
        translate_btn = gr.Button("Translate")
    with gr.Row():
        output = gr.Textbox(label="Translation Output")
    with gr.Row():
        speak_btn = gr.Button("🔊 Speak Translation")
        audio_output = gr.Audio(label="Listen", type="filepath")

    # Events
    translate_btn.click(translate, inputs=[input_text, src, tgt], outputs=output)
    speak_btn.click(speak, inputs=[output, tgt], outputs=audio_output)

# 🟢 Launch
demo.launch(share=True)


✅ Model and tokenizer loaded!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a3c53deb3c961ac8b6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
